In [1]:
import os
from dotenv import load_dotenv
from langchain_core import messages

from schemas.agent_schema import AgentRequest, ModeResponse

load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)

C:\Users\ABDERRAZEK\PycharmProjects\FastAPIAgentParking\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from agent.vectorstore.vectorstore import VectorStore
vectorstore = VectorStore()
vectorstore.setup()


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 16276.84it/s]


Loaded 78 chunks
Loaded Chroma with 78 vectors


EnsembleRetriever(retrievers=[BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000024CE7A6BAA0>, k=5), VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000024CEE680110>, search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 20, 'lambda_mult': 0.8, 'filter': {'type': 'text'}})], weights=[0.4, 0.6])

In [13]:
from agent.graph.graph_builder import GraphBuilder
graph_builder=GraphBuilder(llm=model,vector_store=vectorstore)
result = graph_builder.run_graph(question=" list of USERS?",user_id=10,reclamation_id=4,mode_response=ModeResponse.user_response)
print(result["messages"])


[HumanMessage(content=' list of reservation?', additional_kwargs={}, response_metadata={}, id='7a410258-14fd-47c2-8272-29769c5b7439'), AIMessage(content='', additional_kwargs={'reasoning_content': 'User: "list of reservation?" They want list of reservations. Use filter_reservations_tool with default parameters.', 'tool_calls': [{'id': 'fc_6d1a3f57-29c9-4935-aa8c-bd0ee97ff40b', 'function': {'arguments': '{"limit":20}', 'name': 'filter_reservations_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 1627, 'total_tokens': 1677, 'completion_time': 0.050365973, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.008843804, 'prompt_tokens_details': {'cached_tokens': 1536}, 'queue_time': 0.016033257, 'total_time': 0.059209777}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8e6963b3d5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--0

In [14]:
print(result["answer"])

Please switch to general mode.


In [ ]:
from typing import List
from agent.vectorstore.vectorstore import VectorStore

vectorstore = VectorStore()

from langchain_core.tools import tool
from langchain_core.documents import Document


@tool
def retriever_tool(query: str) -> str:
    """
    Retrieve relevant documents and passages from the vector database
    based on the user query.

    Use this tool when:
    - the user asks questions about indexed documents
    - searching knowledge base content
    - retrieving contextual information for RAG
    - finding relevant passages from PDFs or text files
    """

    vectorstore.setup()

    docs: List[Document] = vectorstore.retrieve(query)

    if not docs:
        return "No documents found."

    merged = []

    for i, d in enumerate(docs[:8], start=1):
        meta = d.metadata if hasattr(d, "metadata") else {}

        title = (
            meta.get("title")
            or meta.get("source")
            or f"doc_{i}"
        )

        merged.append(
            f"[{i}] {title}\n{d.page_content}"
        )

    return "\n\n".join(merged)

In [ ]:


from services.subscription_service import filter_subscriptions
from services.plan_parking_lot_service import filter_plan_parking_lots
from services.plan_service import filter_plans
from services.user_service import filter_users
from services.reclamation_service import filter_reclamations
from datetime import date, datetime
from services.tarif_grid_service import filter_tarif_grids
import json
from typing import Optional
from langchain_core.tools import tool
from services.parking_lot_service import get_parking_lots

def serialize_results(results):
    return json.dumps(
        [r.model_dump(mode="json") for r in results],
        indent=2,
        default=str
    )



@tool
def get_parking_lots_tool(
        id: Optional[int] = None,
        name: Optional[str] = None,
        address: Optional[str] = None,
        city: Optional[str] = None,
        country: Optional[str] = None,
        covered: Optional[bool] = None,
        numberOfPlaces: Optional[int] = None,
        numberOfPlaceAvailable: Optional[int] = None,
        description: Optional[str] = None,
        statusParking: Optional[str] = None,
        reservationAvailability: Optional[bool] = None,
        subscriptionAvailability: Optional[bool] = None,
        tarifGridId: Optional[int] = None,
        skip: int = 0,
        limit: int = 20
) -> str:
    """Filter parking lots by id, name, address, city, country, status, availability, and tariff grid."""

    db = SessionLocal()
    try:
        filters = {
            "id": id,
            "name": name,
            "address": address,
            "city": city,
            "country": country,
            "covered": covered,
            "numberOfPlaces": numberOfPlaces,
            "numberOfPlaceAvailable": numberOfPlaceAvailable,
            "description": description,
            "statusParking": statusParking,
            "reservationAvailability": reservationAvailability,
            "subscriptionAvailability": subscriptionAvailability,
            "tarifGridId": tarifGridId,
        }

        results = get_parking_lots(db, filters, skip, limit)

        return serialize_results(results)
    finally:
        db.close()


@tool
def filter_tarif_grids_tool(
        id: Optional[int] = None,
        name: Optional[str] = None,
        skip: int = 0,
        limit: int = 20
) -> str:
    """Filter tarif grids by id or name."""

    db = SessionLocal()
    try:
        filters = {
            "id": id,
            "name": name
        }

        results = filter_tarif_grids(db, filters, skip, limit)

        return serialize_results(results)
    finally:
        db.close()


from langchain_core.tools import tool
from database import SessionLocal
from services.reservation_service import filter_reservations


@tool
def filter_reservations_tool(
        id: Optional[int] = None,
        userId: Optional[int] = None,
        parkingLotId: Optional[int] = None,
        status: Optional[str] = None,
        totalPrice: Optional[float] = None,
        startDateFrom: Optional[date] = None,
        startDateTo: Optional[date] = None,
        endDateFrom: Optional[date] = None,
        endDateTo: Optional[date] = None,
        entryTimeFrom: Optional[date] = None,
        entryTimeTo: Optional[date] = None,
        skip: int = 0,
        limit: int = 20,
) -> str:
    """Filter reservations by id, user, parking lot, status, price, and date ranges."""

    db = SessionLocal()
    try:
        filters = {
            "id": id,
            "userId": userId,
            "parkingLotId": parkingLotId,
            "status": status,
            "totalPrice": totalPrice,
            "startDateFrom": startDateFrom,
            "startDateTo": startDateTo,
            "endDateFrom": endDateFrom,
            "endDateTo": endDateTo,
            "entryTimeFrom": entryTimeFrom,
            "entryTimeTo": entryTimeTo,
        }

        results = filter_reservations(
            db=db,
            filters=filters,
            skip=skip,
            limit=limit,
        )

        return serialize_results(results)
    finally:
        db.close()

@tool
def filter_users_tool(
        id: int | None = None,
        firstName: str | None = None,
        lastName: str | None = None,
        email: str | None = None,
        phone: str | None = None,
        role: str | None = None,
        accountStatus: str | None = None,
        skip: int = 0,
        limit: int = 20
) -> str:
    """
       Filter users by id, name, email, phone, role, or account status.

       Use this tool when:
       - user asks about users
       - search users by name or email
       - filter by role or status
       """
    db = SessionLocal()

    try:
        filters = {
            "id": id,
            "firstName": firstName,
            "lastName": lastName,
            "email": email,
            "phone": phone,
            "role": role,
            "accountStatus": accountStatus,
        }

        result = filter_users(db, filters, skip, limit)
        return serialize_results(result)

    finally:
        db.close()


@tool
def filter_reclamations_tool(
        id: int | None = None,
        clientId: int | None = None,
        adminId: int | None = None,
        status: str | None = None,
        subject: str | None = None,
        content: str | None = None,
        solution: str | None = None,
        skip: int = 0,
        limit: int = 20
) -> str:
    """Filter reclamations by id, client, admin, status, subject, content, or solution."""

    db = SessionLocal()
    try:
        filters = {
            "id": id,
            "clientId": clientId,
            "adminId": adminId,
            "status": status,
            "subject": subject,
            "content": content,
            "solution": solution,
        }

        results = filter_reclamations(db, filters, skip, limit)
        return serialize_results(results)

    finally:
        db.close()


@tool
def filter_plans_tool(
        id: Optional[int] = None,
        name: Optional[str] = None,
        NumberOfBenefitDays: Optional[int] = None,
        startDateFrom: Optional[datetime] = None,
        startDateTo: Optional[datetime] = None,
        endDateFrom: Optional[datetime] = None,
        endDateTo: Optional[datetime] = None,
        isActive: Optional[bool] = None,
        skip: int = 0,
        limit: int = 20
) -> str:
    """Filter plans by name, benefit days, and date ranges."""

    db = SessionLocal()
    try:
        filters = {
            "id": id,
            "name": name,
            "NumberOfBenefitDays": NumberOfBenefitDays,
            "startDateFrom": startDateFrom,
            "startDateTo": startDateTo,
            "endDateFrom": endDateFrom,
            "endDateTo": endDateTo,
            "isActive": isActive
        }

        results = filter_plans(db, filters, skip, limit)

        return json.dumps(
            [r.model_dump(mode="json") for r in results],
            indent=2,
            default=str
        )
    finally:
        db.close()


from typing import Optional, List
from langchain_core.tools import tool


@tool
def filter_plan_parking_lots_tool(
        id: Optional[int] = None,
        planId: Optional[int] = None,

        # CHANGED: int -> List[int]
        parkingLotId: Optional[List[int]] = None,

        status: Optional[str] = None,
        renewFee: Optional[float] = None,
        subscriptionFee: Optional[float] = None,
        renewFeeMin: Optional[float] = None,
        renewFeeMax: Optional[float] = None,
        subscriptionFeeMin: Optional[float] = None,
        subscriptionFeeMax: Optional[float] = None,
        skip: int = 0,
        limit: int = 20
) -> str:
    """
    Filter plan parking lots by plan, parking lot, status,
    renew fee, and subscription fee.

    parkingLotId supports multiple IDs:
    example -> [3, 4, 5, 7]
    """

    db = SessionLocal()
    try:
        filters = {
            "id": id,
            "planId": planId,
            "parkingLotId": parkingLotId,  # now list supported
            "status": status,
            "renewFee": renewFee,
            "subscriptionFee": subscriptionFee,
            "renewFeeMin": renewFeeMin,
            "renewFeeMax": renewFeeMax,
            "subscriptionFeeMin": subscriptionFeeMin,
            "subscriptionFeeMax": subscriptionFeeMax,
        }

        results = filter_plan_parking_lots(
            db,
            filters,
            skip,
            limit
        )

        return serialize_results(results)

    finally:
        db.close()


@tool
def filter_subscriptions_tool(
        id: Optional[int] = None,
        status: Optional[str] = None,
        planParkingLotId: Optional[int] = None,
        userId: Optional[int] = None,
        startDateFrom: Optional[datetime] = None,
        startDateTo: Optional[datetime] = None,
        endDateFrom: Optional[datetime] = None,
        endDateTo: Optional[datetime] = None,
        isActive: Optional[bool] = None,
        userEmail: Optional[str] = None,
        userName: Optional[str] = None,
        skip: int = 0,
        limit: int = 20
) -> str:
    """Filter subscriptions by id, status, user, plan parking lot, dates, and active state."""

    db = SessionLocal()
    try:
        filters = {
            "id": id,
            "status": status,
            "planParkingLotId": planParkingLotId,
            "userId": userId,
            "startDateFrom": startDateFrom,
            "startDateTo": startDateTo,
            "endDateFrom": endDateFrom,
            "endDateTo": endDateTo,
            "isActive": isActive,
            "userEmail": userEmail,
            "userName": userName,
        }

        results = filter_subscriptions(db, filters, skip, limit)

        return serialize_results(results)
    finally:
        db.close()
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def unsupported_request(reason: str) -> str:
    """Use this tool when the user asks for something that is خارج available tools."""
    return "This request is not supported by the available tools."




In [ ]:
tools = [
    unsupported_request,
    filter_tarif_grids_tool,
    filter_reclamations_tool,
    filter_users_tool,
    filter_reservations_tool,
    get_parking_lots_tool,
    filter_plans_tool,
    filter_plan_parking_lots_tool,
    filter_subscriptions_tool,
    retriever_tool
]


In [ ]:
from langchain.agents import create_agent
system_prompt = """
You are a tool-using assistant.

RULES:
- You must ALWAYS use at least one tool.
- Never answer without tool usage.
- If no suitable tool exists, use unsupported_request.
- Do not rely on internal knowledge.
"""
agent = create_agent(model, tools=tools,system_prompt=system_prompt
)
agent

In [15]:
from langchain_core.messages import AIMessage, ToolMessage

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "when many reservation clinet named makrem    can  refound  money "
            }
        ]
    }
)

for msg in response["messages"]:

    if isinstance(msg, AIMessage):
        print("AI:")
        print(msg.content)

        # Tool calls
        if msg.tool_calls:
            print("Tools Used:")
            for tool in msg.tool_calls:
                print(tool["name"])
                print(tool["args"])

    elif isinstance(msg, ToolMessage):
        print("Tool Result:")
        print(msg.content)

    print("---------------")

NameError: name 'agent' is not defined